# Cleaned household clustering pipeline

This notebook combines the feature-building notebook and the refined two-stage clustering notebook into one safer end-to-end workflow.

## What it does
1. Load raw wide time-series data
2. Remove all-zero series
3. Build a clean series matrix
4. Extract interpretable time-series features
5. Optionally add catch22 features
6. Clean and stabilize the feature matrix
7. Split households into sparsity buckets using `zero_fraction`
8. Compare KMeans and GMM within each bucket across multiple `k`
9. Pick the best model per bucket using silhouette score
10. Save feature and cluster outputs


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from scipy.stats import linregress
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

plt.rcParams["figure.figsize"] = (10, 5)
RANDOM_STATE = 42


## 1. Load raw sample data

In [ ]:
file_2023 = Path("../data/raw/sample_23.csv")
data_2023 = pd.read_csv(file_2023)

display(data_2023.head())
print("Wide shape:", data_2023.shape)


## 2. Reshape to ID / Date / Value and remove all-zero series

In [ ]:
date_columns = [col for col in data_2023.columns if col != "ID"]

long_df = data_2023.melt(
    id_vars="ID",
    value_vars=date_columns,
    var_name="Date",
    value_name="Value",
)

try:
    long_df["Date"] = pd.to_datetime(long_df["Date"])
    print("Date column parsed successfully.")
except Exception:
    print("Could not parse Date column automatically. Keeping as string.")

display(long_df.head())
print("Long shape:", long_df.shape)

id_all_zero = long_df.groupby("ID")["Value"].apply(lambda x: (x == 0).all())
print("Number of all-zero IDs:", int(id_all_zero.sum()))

valid_ids = id_all_zero.index[~id_all_zero]
filtered_long_df = long_df[long_df["ID"].isin(valid_ids)].reset_index(drop=True)

print("Filtered long shape:", filtered_long_df.shape)
display(filtered_long_df.head())


## 3. Pivot back to a clean time-series matrix

In [ ]:
series_matrix = filtered_long_df.pivot(index="ID", columns="Date", values="Value")

try:
    series_matrix = series_matrix.sort_index(axis=1)
except Exception:
    pass

display(series_matrix.head())
print("Series matrix shape:", series_matrix.shape)
print("Missing values total:", int(series_matrix.isna().sum().sum()))

# Fill any missing values conservatively with 0 after pivot.
series_matrix = series_matrix.fillna(0.0)


## 4. Feature helper functions

In [ ]:
def safe_mean(x):
    return float(np.mean(x)) if len(x) > 0 else 0.0


def safe_std(x):
    return float(np.std(x, ddof=0)) if len(x) > 0 else 0.0


def coefficient_of_variation(x):
    m = np.mean(x)
    s = np.std(x, ddof=0)
    return float(s / m) if abs(m) > 1e-8 else 0.0


def autocorr_lag(x, lag):
    x = np.asarray(x, dtype=float)
    if len(x) <= lag:
        return 0.0
    x1 = x[:-lag]
    x2 = x[lag:]
    if np.std(x1) < 1e-8 or np.std(x2) < 1e-8:
        return 0.0
    return float(np.corrcoef(x1, x2)[0, 1])


def trend_slope(x):
    x = np.asarray(x, dtype=float)
    t = np.arange(len(x))
    if np.std(x) < 1e-8:
        return 0.0
    return float(linregress(t, x).slope)


def zero_runs(x):
    runs = []
    run_len = 0
    for val in x:
        if val == 0:
            run_len += 1
        else:
            if run_len > 0:
                runs.append(run_len)
                run_len = 0
    if run_len > 0:
        runs.append(run_len)
    return runs


def nonzero_stats(x):
    nz = x[x != 0]
    if len(nz) == 0:
        return 0.0, 0.0, 0.0
    return float(np.mean(nz)), float(np.std(nz, ddof=0)), float(np.median(nz))


def rolling_features(x, window=7):
    s = pd.Series(x)
    roll_mean = s.rolling(window=window, min_periods=1).mean()
    roll_std = s.rolling(window=window, min_periods=1).std().fillna(0.0)
    return float(roll_mean.std()), float(roll_std.mean())


def burstiness(x):
    x = np.asarray(x, dtype=float)
    m = np.mean(x)
    s = np.std(x, ddof=0)
    if s + m < 1e-8:
        return 0.0
    return float((s - m) / (s + m))


def peak_features(x):
    x = np.asarray(x, dtype=float)
    if len(x) < 3:
        return 0.0, 0.0
    peaks = []
    for i in range(1, len(x) - 1):
        if x[i] > x[i - 1] and x[i] > x[i + 1]:
            peaks.append(i)
    n_peaks = len(peaks)
    if n_peaks < 2:
        return float(n_peaks), 0.0
    avg_peak_spacing = float(np.mean(np.diff(peaks)))
    return float(n_peaks), avg_peak_spacing


def weekday_profile(x, dates):
    try:
        weekdays = pd.to_datetime(dates).weekday
    except Exception:
        return [0.0] * 7

    x = np.asarray(x, dtype=float)
    overall_mean = np.mean(x)
    if abs(overall_mean) < 1e-8:
        overall_mean = 1.0

    feats = []
    for d in range(7):
        vals = x[weekdays == d]
        feats.append(float(np.mean(vals) / overall_mean) if len(vals) > 0 else 0.0)
    return feats


def seasonal_peak_distribution(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return 0.0, 0.0
    threshold = np.quantile(x, 0.9)
    top_idx = np.where(x >= threshold)[0]
    if len(top_idx) == 0:
        return 0.0, 0.0
    first_half = np.mean(top_idx < len(x) / 2)
    second_half = np.mean(top_idx >= len(x) / 2)
    return float(first_half), float(second_half)


## 5. Extract custom features

In [ ]:
def extract_features_from_series_matrix(series_matrix):
    feature_rows = []
    dates = np.array(series_matrix.columns)

    for idx, row in series_matrix.iterrows():
        x = row.to_numpy(dtype=float)

        mean_val = safe_mean(x)
        median_val = float(np.median(x))
        std_val = safe_std(x)
        cv_val = coefficient_of_variation(x)

        p10 = float(np.quantile(x, 0.10))
        p90 = float(np.quantile(x, 0.90))
        p90_p10_ratio = float(p90 / (p10 + 1e-8)) if abs(p10) > 1e-8 else 0.0

        max_val = float(np.max(x))
        max_median_ratio = float(max_val / (median_val + 1e-8)) if abs(median_val) > 1e-8 else 0.0

        above_p90_fraction = float(np.mean(x > p90)) if len(x) > 0 else 0.0

        slope = trend_slope(x)
        acf1 = autocorr_lag(x, 1)
        acf7 = autocorr_lag(x, 7)
        acf14 = autocorr_lag(x, 14)

        zero_fraction = float(np.mean(x == 0))
        nonzero_fraction = float(np.mean(x != 0))

        zruns = zero_runs(x)
        num_zero_runs = float(len(zruns))
        avg_zero_run_length = float(np.mean(zruns)) if len(zruns) > 0 else 0.0
        max_zero_run_length = float(np.max(zruns)) if len(zruns) > 0 else 0.0

        nonzero_mean, nonzero_std, nonzero_median = nonzero_stats(x)

        rolling7_mean_std, rolling7_std_mean = rolling_features(x, window=7)

        weekday_feats = weekday_profile(x, dates)
        weekday_weekend_gap = float(
            np.mean(weekday_feats[:5]) - np.mean(weekday_feats[5:])
        )

        burst = burstiness(x)
        n_peaks, avg_peak_spacing = peak_features(x)

        top10_first_half, top10_second_half = seasonal_peak_distribution(x)

        feature_rows.append({
            "ID": idx,
            "mean": mean_val,
            "median": median_val,
            "std": std_val,
            "cv": cv_val,
            "p90_p10_ratio": p90_p10_ratio,
            "max_median_ratio": max_median_ratio,
            "above_p90_fraction": above_p90_fraction,
            "trend_slope": slope,
            "acf1": acf1,
            "acf7": acf7,
            "acf14": acf14,
            "zero_fraction": zero_fraction,
            "nonzero_fraction": nonzero_fraction,
            "num_zero_runs": num_zero_runs,
            "avg_zero_run_length": avg_zero_run_length,
            "max_zero_run_length": max_zero_run_length,
            "nonzero_mean": nonzero_mean,
            "nonzero_std": nonzero_std,
            "nonzero_median": nonzero_median,
            "rolling7_mean_std": rolling7_mean_std,
            "rolling7_std_mean": rolling7_std_mean,
            "weekday_0_norm": weekday_feats[0],
            "weekday_1_norm": weekday_feats[1],
            "weekday_2_norm": weekday_feats[2],
            "weekday_3_norm": weekday_feats[3],
            "weekday_4_norm": weekday_feats[4],
            "weekday_5_norm": weekday_feats[5],
            "weekday_6_norm": weekday_feats[6],
            "weekday_weekend_gap": weekday_weekend_gap,
            "burstiness": burst,
            "num_peaks": n_peaks,
            "avg_peak_spacing": avg_peak_spacing,
            "top10_first_half": top10_first_half,
            "top10_second_half": top10_second_half,
        })

    return pd.DataFrame(feature_rows)


feature_df = extract_features_from_series_matrix(series_matrix)

display(feature_df.head())
print("Feature table shape:", feature_df.shape)

#output_file = Path("../data/processed/feature_based_clusters_2023_refined.csv")
#feature_df.to_csv(output_file, index=False)

## 6. Optionally add catch22 features

In [ ]:
USE_CATCH22 = True

if USE_CATCH22:
    try:
        import pycatch22

        catch22_rows = []
        for idx, row in series_matrix.iterrows():
            x = row.to_numpy(dtype=float)
            feats = pycatch22.catch22_all(x)
            row_dict = {"ID": idx}
            for name, value in zip(feats["names"], feats["values"]):
                row_dict[f"catch22_{name}"] = value
            catch22_rows.append(row_dict)

        catch22_df = pd.DataFrame(catch22_rows)
        feature_df = feature_df.merge(catch22_df, on="ID", how="left")
        print("catch22 features added.")
    except ImportError:
        print("pycatch22 is not installed. Skipping catch22 features.")

print("Feature table shape after optional catch22:", feature_df.shape)


## 7. Clean and stabilize the feature matrix

In [ ]:
def winsorize_series(s, lower_q=0.01, upper_q=0.99):
    lower = s.quantile(lower_q)
    upper = s.quantile(upper_q)
    return s.clip(lower=lower, upper=upper)


def is_heavy_tailed(s, ratio_threshold=10, skew_threshold=2):
    s = s.dropna()
    if len(s) == 0:
        return False
    q50 = s.quantile(0.50)
    q99 = s.quantile(0.99)
    ratio = (q99 + 1e-12) / (abs(q50) + 1e-12)
    skew = s.skew() if len(s) > 2 else 0
    return (ratio >= ratio_threshold) or (abs(skew) >= skew_threshold)


feature_df = feature_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

exclude_cols = [
    "ID",
    "Cluster",
    "SparsityGroup",
    "SparsityBucket",
    "RefinedClusterLocal",
    "RefinedCluster",
]
feature_cols = [c for c in feature_df.columns if c not in exclude_cols]

feature_df_clean = feature_df.copy()

heavy_tail_cols = []
for col in feature_cols:
    if pd.api.types.is_numeric_dtype(feature_df_clean[col]) and is_heavy_tailed(feature_df_clean[col]):
        heavy_tail_cols.append(col)

print("Heavy-tailed columns detected:", len(heavy_tail_cols))
print(heavy_tail_cols[:20])

for col in heavy_tail_cols:
    feature_df_clean[col] = winsorize_series(feature_df_clean[col], 0.01, 0.99)
    if (feature_df_clean[col] >= 0).all():
        feature_df_clean[col] = np.log1p(feature_df_clean[col])

display(feature_df_clean.head())


## 8. Build sparsity buckets

In [ ]:
assert "zero_fraction" in feature_df_clean.columns, "zero_fraction column is required"

def sparsity_bucket(z):
    if z < 0.05:
        return "dense"
    elif z < 0.25:
        return "moderate_zero"
    else:
        return "high_zero"

feature_df_clean["SparsityBucket"] = feature_df_clean["zero_fraction"].apply(sparsity_bucket)

display(feature_df_clean["SparsityBucket"].value_counts())


## 9. Helper functions for two-stage clustering

In [ ]:
def get_feature_matrix(df):
    exclude = {
        "ID",
        "Cluster",
        "SparsityGroup",
        "SparsityBucket",
        "RefinedClusterLocal",
        "RefinedCluster",
    }
    cols = [
        c for c in df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
    ]
    X = df[cols].copy()
    return X, cols


def safe_silhouette(X_scaled, labels, random_state=RANDOM_STATE):
    unique_labels = np.unique(labels)
    if len(unique_labels) < 2:
        return -1.0
    if len(unique_labels) >= len(X_scaled):
        return -1.0
    try:
        return float(
            silhouette_score(
                X_scaled,
                labels,
                sample_size=min(5000, len(X_scaled)),
                random_state=random_state,
            )
        )
    except Exception:
        return -1.0


def evaluate_kmeans(X_scaled, k, random_state=RANDOM_STATE):
    if k >= len(X_scaled):
        return None, None, -1.0
    model = KMeans(n_clusters=k, random_state=random_state, n_init=20)
    labels = model.fit_predict(X_scaled)
    sil = safe_silhouette(X_scaled, labels, random_state=random_state)
    return model, labels, sil


def evaluate_gmm(X_scaled, k, random_state=RANDOM_STATE):
    if k >= len(X_scaled):
        return None, None, -1.0
    model = GaussianMixture(
        n_components=k,
        covariance_type="diag",
        random_state=random_state,
        n_init=3,
        reg_covar=1e-6,
    )
    labels = model.fit_predict(X_scaled)
    sil = safe_silhouette(X_scaled, labels, random_state=random_state)
    return model, labels, sil


def cluster_size_table(labels):
    s = pd.Series(labels).value_counts().sort_index()
    return pd.DataFrame({
        "cluster": s.index,
        "count": s.values,
        "pct": 100 * s.values / s.values.sum()
    })


## 10. Run two-stage clustering by sparsity bucket

In [ ]:
k_values = [3, 4, 5, 6, 8]
results = []
best_models = {}

for bucket_name, bucket_df in feature_df_clean.groupby("SparsityBucket"):
    print(f"\n=== Bucket: {bucket_name} | n={len(bucket_df)} ===")

    X_bucket, feat_cols = get_feature_matrix(bucket_df)

    if len(bucket_df) < 10 or X_bucket.shape[1] == 0:
        print("Skipping bucket because it is too small or has no usable features.")
        continue

    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X_bucket)

    bucket_results = []

    for k in k_values:
        if k >= len(bucket_df):
            continue

        km_model, km_labels, km_sil = evaluate_kmeans(X_scaled, k)
        bucket_results.append({
            "bucket": bucket_name,
            "method": "kmeans",
            "k": k,
            "silhouette": km_sil
        })

        gmm_model, gmm_labels, gmm_sil = evaluate_gmm(X_scaled, k)
        bucket_results.append({
            "bucket": bucket_name,
            "method": "gmm",
            "k": k,
            "silhouette": gmm_sil
        })

    bucket_results_df = pd.DataFrame(bucket_results).sort_values("silhouette", ascending=False)
    display(bucket_results_df)

    if bucket_results_df.empty:
        print("No valid models for this bucket.")
        continue

    best_row = bucket_results_df.iloc[0]
    best_method = best_row["method"]
    best_k = int(best_row["k"])

    print(f"Best for {bucket_name}: {best_method}, k={best_k}, silhouette={best_row['silhouette']:.4f}")

    if best_method == "kmeans":
        best_model, best_labels, best_sil = evaluate_kmeans(X_scaled, best_k)
    else:
        best_model, best_labels, best_sil = evaluate_gmm(X_scaled, best_k)

    best_models[bucket_name] = {
        "model": best_model,
        "labels": best_labels,
        "scaler": scaler,
        "feature_cols": feat_cols,
        "silhouette": best_sil,
        "method": best_method,
        "k": best_k,
        "index": bucket_df.index
    }

    results.append(bucket_results_df)

results_df = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
display(results_df.sort_values(["bucket", "silhouette"], ascending=[True, False]) if not results_df.empty else results_df)


## 11. Write refined cluster labels back

In [ ]:
feature_df_clean["RefinedClusterLocal"] = -999
feature_df_clean["RefinedCluster"] = None

for bucket_name, info in best_models.items():
    idx = info["index"]
    labels = info["labels"]

    feature_df_clean.loc[idx, "RefinedClusterLocal"] = labels
    feature_df_clean.loc[idx, "RefinedCluster"] = [
        f"{bucket_name}_{int(l)}" for l in labels
    ]

display(feature_df_clean[["ID", "SparsityBucket", "RefinedClusterLocal", "RefinedCluster"]].head())


## 12. Inspect refined cluster sizes

In [ ]:
cluster_sizes = feature_df_clean["RefinedCluster"].value_counts(dropna=False).reset_index()
cluster_sizes.columns = ["Cluster", "Count"]
cluster_sizes["Pct"] = 100 * cluster_sizes["Count"] / cluster_sizes["Count"].sum()

display(cluster_sizes.sort_values("Count", ascending=False))

plt.figure(figsize=(12, 4))
plt.bar(cluster_sizes["Cluster"].astype(str), cluster_sizes["Count"])
plt.xticks(rotation=90)
plt.title("Refined cluster sizes")
plt.ylabel("Number of households")
plt.tight_layout()
plt.show()

print("Cluster counts:")
for cluster, count in feature_df_clean["RefinedCluster"].value_counts().sort_index().items():
    print(f"{cluster}: {count} households")


## 13. Summarize refined clusters

In [ ]:
summary_cols = [
    c for c in [
        "mean",
        "std",
        "cv",
        "zero_fraction",
        "avg_zero_run_length",
        "acf7",
        "trend_slope",
        "weekday_weekend_gap",
        "burstiness",
        "num_peaks",
    ]
    if c in feature_df_clean.columns
]

cluster_summary = feature_df_clean.groupby("RefinedCluster")[summary_cols].mean()
display(cluster_summary.sort_index())


## 14. PCA visualization

In [ ]:
X_all, feat_cols = get_feature_matrix(feature_df_clean)
scaler_all = RobustScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X_all_scaled)

plot_df = pd.DataFrame({
    "PC1": X_2d[:, 0],
    "PC2": X_2d[:, 1],
    "Cluster": feature_df_clean["RefinedCluster"].values,
})

sample_plot_df = plot_df.sample(min(12000, len(plot_df)), random_state=RANDOM_STATE)

cluster_to_int = {
    c: i for i, c in enumerate(sorted(sample_plot_df["Cluster"].dropna().unique()))
}
colors = sample_plot_df["Cluster"].map(cluster_to_int)

plt.figure(figsize=(10, 6))
plt.scatter(sample_plot_df["PC1"], sample_plot_df["PC2"], c=colors, s=8, alpha=0.6)
plt.title("PCA view of refined clusters")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()


## 15. Save outputs

In [ ]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

feature_output = output_dir / "feature_based_clusters_2023_refined.csv"
results_output = output_dir / "feature_based_cluster_model_selection.csv"

feature_df_clean.to_csv(feature_output, index=False)
if not results_df.empty:
    results_df.to_csv(results_output, index=False)

print("Saved refined clustering result to:")
print(feature_output)

if not results_df.empty:
    print("Saved model comparison table to:")
    print(results_output)


## Notes

- `RefinedClusterLocal` is only local within each sparsity bucket.
- `RefinedCluster` is the globally interpretable label to use in reporting.
- The PCA plot is for visualization only.
- If `pycatch22` is not installed, the notebook still runs without it.
